# P0 · Anomaly replication on VisA (preregistered)

Tests whether the MVTec AD result holds on a second dataset, with every setting fixed in advance:
see [`docs/preregistration_visa.md`](https://github.com/M3R2T2L2/bovw_forensics/blob/main/docs/preregistration_visa.md).
**Primary image score: mean of the top-10 patch scores.**

| ID | Prediction |
|---|---|
| P1 | codebook AUPRO > coreset AUPRO at M = 64, 256, 1024 |
| P2 | codebook top-10 image AUROC >= coreset - 0.005 at M = 1024 |
| P3 | codebook pixel AUROC >= coreset at M = 1024 |
| P4 | codebook within 0.01 of the full bank at K = 1024 (image top-10, pixel, AUPRO) |

**Data:** VisA (Zou et al., ECCV 2022), CC BY 4.0, official 1-class split; 12 categories, 10,821 images.
**Runtime (T4):** about 2 h, plus a one-time ~1.8 GB download to Drive. Rerun after a disconnect to resume.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data
The archive is kept on Drive so it downloads only once; each new runtime extracts it locally.

In [ ]:
import os, subprocess
from bovw import ad_data
DRIVE_TAR = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar"
LOCAL = "/content/visa"

if not os.path.isfile(DRIVE_TAR) or os.path.getsize(DRIVE_TAR) < 1e9:
    ad_data.ensure_archive(DRIVE_TAR, url=ad_data.VISA_URL, sha256=None)
if not os.path.isdir(LOCAL):
    os.makedirs(LOCAL)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", LOCAL], check=True)
VISA_ROOT = ad_data.find_visa_root(LOCAL)
print(VISA_ROOT, sorted(c for c in os.listdir(VISA_ROOT) if c in ad_data.VISA_CATEGORIES))
print("archive SHA-256 (record in the paper):", ad_data.sha256sum(DRIVE_TAR))

## 3 · Run

In [ ]:
from bovw.sweep import load_config
from bovw import anomaly

cfg = load_config("configs/p0_anomaly_visa.yaml")
cfg["data"]["root"] = VISA_ROOT
df = anomaly.run(cfg)

## 4 · Preregistered checks

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
checks = anomaly.prereg_checks(df)
checks.to_csv(f"{cfg['results_dir']}/{cfg['name']}_prereg.csv", index=False)
checks

## 5 · Full table (secondary metrics are exploratory)

In [ ]:
t = anomaly.followup_table(df)
t.to_csv(f"{cfg['results_dir']}/{cfg['name']}_followup.csv")
t.round(3)

In [ ]:
d = df[df.k.fillna(0).astype(int).isin([0, 1024])]
pd.concat({m: d.pivot_table(index="category", columns="method", values=m)
           for m in ["image_auroc_top10", "pixel_auroc", "aupro"]}, axis=1).round(3)